# RK T4 Flagship Rev 4.0 — 1-Click Cloud Hardware Verification Suite (Google Colab)

This notebook executes the **entire zero-shortcut hardware verification pipeline** for **RK T4 Rev 4.0** (`sc8z35a-collab/GIT-Y5`) on Google Colab:
1. **Install KiCad 9 (`kicad-cli` + `pcbnew`) & Node.js 24 LTS**
2. **Netlist Generation & 111-Check ERC Suite** (`generate_board.mjs` + `verify_erc.mjs`)
3. **Full 14-Layer 1.20 mm 3+8+3 Staggered HDI Mainboard Routing & Timing Closure** (`route_hdi_rev4.mjs`)
4. **Full 8-Layer 0.80 mm JLC08081H-1080A 5G RF Mezzanine Routing** (`route_mezzanine_rf.mjs`)
5. **Strict Native KiCad 9 DRC (`--severity-all --all-track-errors --exit-code-violations`) & Gerber/Drill/BOM/CPL Export** (`export_mfg.mjs`)
6. **3D CAD Single-File Build & Headless Chromium Box3 Zero-Interference Audit** (`check_3d_intersections.mjs`)

Click **Runtime -> Run all** to execute all checks and download the verified Gerber ZIPs.

In [ ]:
%%bash
set -euo pipefail
echo "=== 1. Installing KiCad 9 & Node.js 24 LTS ==="
apt-get update -qq
apt-get install -y -qq software-properties-common curl ca-certificates gnupg chromium-browser
add-apt-repository -y ppa:kicad/kicad-9.0-releases
curl -fsSL https://deb.nodesource.com/setup_24.x | bash -
apt-get update -qq
apt-get install -y -qq kicad nodejs
echo "Node: $(node -v) | KiCad CLI: $(kicad-cli --version)"

In [ ]:
%%bash
set -euo pipefail
rm -rf /content/GIT-Y5
git clone --depth 1 https://github.com/sc8z35a-collab/GIT-Y5.git /content/GIT-Y5
cd /content/GIT-Y5
echo "=== 2. Netlist & 111-Check ERC Verification ==="
node hardware/generate_board.mjs
node hardware/verify_erc.mjs

In [ ]:
%%bash
set -euo pipefail
cd /content/GIT-Y5
echo "=== 3. Routing 14L 1.20mm 3+8+3 Staggered HDI Mainboard (Rev 4.0) ==="
node hardware/route_hdi_rev4.mjs
echo "=== 4. Routing 8L 0.80mm JLC08081H-1080A RF Mezzanine (Rev 4.0) ==="
node hardware/route_mezzanine_rf.mjs

In [ ]:
%%bash
set -euo pipefail
cd /content/GIT-Y5
echo "=== 5. Strict Official KiCad 9 Native DRC & Manufacturing Export ==="
node hardware/export_mfg.mjs
echo "=== 6. 3D CAD Build & Box3 Physical Interference Audit ==="
npm ci --silent
npm run build
node hardware/check_3d_intersections.mjs "file:///content/GIT-Y5/index.html"

In [ ]:
import json
from pathlib import Path

root = Path('/content/GIT-Y5/hardware')
main_drc = json.loads((root / 'kicad_drc_report.json').read_text())
mezz_drc = json.loads((root / 'kicad_mezz_drc_report.json').read_text())
lm = json.loads((root / 'length_match_report.json').read_text())
cad = json.loads((root / '3d_interference_report.json').read_text())

print('=== RK T4 REV 4.0 VERIFICATION SUMMARY ===')
print(f"Mainboard 14L DRC : violations={len(main_drc['violations'])}, unconnected={len(main_drc['unconnected_items'])}")
print(f"Mezzanine  8L DRC : violations={len(mezz_drc['violations'])}, unconnected={len(mezz_drc['unconnected_items'])}")
print(f"3D CAD Clearance  : {cad['totalInterferences']} interferences across {cad['totalSolidMeshes']} meshes ({cad['bodyThicknessMm']} mm body)")
print('\nHigh-Speed Timing Closure:')
for g in lm['groups']:
    if g['type'].startswith('LPDDR5X') or g['type'] == 'eMMC-5.1':
        print(f"  {g['group']:24s} {g['routedCount']:2d}/{g['netCount']:2d}  skew={g['afterSkewMm']:.3f} mm ({g['afterSkewPs']:.2f} ps)")

try:
    from google.colab import files
    print('\nReady to download Gerbers — uncomment lines below in Colab:')
    print('# files.download("/content/GIT-Y5/hardware/mfg/rk_t4_mainboard_gerbers.zip")')
    print('# files.download("/content/GIT-Y5/hardware/mfg/rk_t4_mezzanine_gerbers.zip")')
except ImportError:
    pass